# TASK 3 · Cleaning Data

## Objective
The objective of this project is to clean a deliberately messy dataset and transform it into a clean, consistent, and analysis-ready dataset.

The cleaning process includes:
- Data quality assessment
- Missing value handling
- Duplicate removal
- Standardisation
- Outlier detection
- Data type correction
- Before vs. after comparison
- Exporting the cleaned dataset

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("dirty_cafe_sales.csv")

In [ ]:
df.head()

In [ ]:
df.shape
df.columns

In [ ]:
df_original = df.copy()

## 1. Initial Dataset Inspection

The dataset is first inspected to understand its structure, dimensions, column names, data types, missing values, and potential inconsistencies.

In [ ]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

## 2. Missing Value Analysis

Missing values are identified for every column before cleaning. The number and percentage of missing observations are calculated to determine the appropriate treatment strategy.

In [ ]:
missing_report = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_report

In [ ]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns

numeric_columns

## 3. Data Quality Report

The data quality report evaluates the raw dataset for missing values, duplicate records, data types, and potential numerical range anomalies. This report provides a baseline for evaluating the effectiveness of the cleaning process.

In [ ]:
quality_report = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isnull().sum(),
    "Missing %": (df.isnull().sum() / len(df) * 100).round(2),
    "Unique Values": df.nunique()
})

quality_report

In [ ]:
print("Total duplicate rows:", df.duplicated().sum())

## 4. Missing Data Handling Strategy

Different strategies were selected according to the nature and amount of missing data.

### Age
Missing Age values are replaced using the median. Age is a numerical variable and the median is less sensitive to extreme values than the mean.

### Embarked
Missing Embarked values are replaced using the mode because Embarked is a categorical variable and only a small number of observations are missing.

### Cabin
The Cabin column contains a large proportion of missing values. Since reliable imputation would not be appropriate, the column is removed rather than introducing potentially misleading information.

This column-specific approach preserves as much useful information as possible while avoiding unreliable assumptions.

In [ ]:
# Check the most frequent value in categorical columns

categorical_columns = ["Item", "Payment Method", "Location"]

for col in categorical_columns:
    print(col, ":", df[col].mode()[0])

In [ ]:
# Fill missing categorical values with mode

for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
# Check missing values after categorical treatment

df[categorical_columns].isnull().sum()

In [ ]:
# Remove rows with missing transaction dates

df = df.dropna(subset=["Transaction Date"])

In [ ]:
# Check dataset size after missing value treatment

print("Rows after missing value treatment:", len(df))

## 5. Duplicate Removal

Duplicate rows can lead to incorrect analysis and inflated results. Exact duplicate records are identified and removed from the dataset.

In [ ]:
# Count duplicate rows before removal

duplicates_before = df.duplicated().sum()

print("Duplicate rows before removal:", duplicates_before)

## 6. Standardisation

Inconsistent formatting can create multiple categories for the same value. Text fields are standardized by removing unnecessary spaces and converting unknown values into missing values.

In [ ]:
# Remove leading and trailing spaces from text columns

text_columns = ["Item", "Payment Method", "Location"]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

In [ ]:
# Convert UNKNOWN values into missing values

unknown_values = ["UNKNOWN", "Unknown", "unknown", "N/A", "NA", ""]

for col in text_columns:
    df[col] = df[col].replace(unknown_values, np.nan)

In [ ]:
# Check unique values after standardisation

for col in text_columns:
    print("\n", col)
    print(df[col].value_counts(dropna=False))

In [ ]:
# Fill newly created missing categorical values with mode

for col in text_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

## 7. Data Type Correction

The dataset contains several columns stored with incorrect data types. Numerical columns are converted to numeric format, the transaction date is converted to datetime format, and the transaction ID is treated as a string identifier.

In [ ]:
# Convert numerical columns to numeric data types

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
# Convert Transaction Date to datetime

df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

In [ ]:
# Convert Transaction ID to string

df["Transaction ID"] = df["Transaction ID"].astype(str)

In [ ]:
# Check data types after correction

df.dtypes

## 8. Value Range and Anomaly Check

Numerical columns are checked for invalid values such as negative quantities, negative prices, or negative total spending.

In [ ]:
df[numeric_columns].describe()

In [ ]:
# Check negative values

for col in numeric_columns:
    print(col, "negative values:", (df[col] < 0).sum())

In [ ]:
# Remove records containing negative numerical values

df = df[df["Quantity"] >= 0]
df = df[df["Price Per Unit"] >= 0]
df = df[df["Total Spent"] >= 0]

In [ ]:
df[numeric_columns].describe()

## 9. Total Spent Consistency Check

The `Total Spent` column is compared with the expected value calculated from `Quantity × Price Per Unit`. This helps identify potentially inconsistent transaction totals.

In [ ]:
# Calculate expected total spent

df["Calculated Total"] = df["Quantity"] * df["Price Per Unit"]

In [ ]:
# Calculate difference between recorded and calculated total

df["Total Difference"] = (
    df["Total Spent"] - df["Calculated Total"]
)

df[["Quantity", "Price Per Unit", "Total Spent",
    "Calculated Total", "Total Difference"]].head(10)

In [ ]:
# Count records where Total Spent differs from calculated total

(df["Total Difference"].abs() > 0.01).sum()

In [ ]:
# Define function to detect outliers using IQR

def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[
        (data[column] < lower_bound) |
        (data[column] > upper_bound)
    ]
    
    return lower_bound, upper_bound, len(outliers)

In [ ]:
# Detect outliers in numerical columns

for col in numeric_columns:
    lower, upper, count = detect_outliers_iqr(df, col)
    
    print("\nColumn:", col)
    print("Lower Bound:", lower)
    print("Upper Bound:", upper)
    print("Number of Outliers:", count)

### Outlier Treatment Decision

Potential outliers were identified using the IQR method. Outliers are not automatically removed because unusually large quantities, prices, or transaction totals may represent legitimate business transactions. Clearly invalid negative values are removed, while legitimate extreme values are retained.

In [ ]:
print("Original rows:", len(df_original))
print("Final rows:", len(df))
print("Missing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())

In [ ]:
df.to_csv("cleaned_cafe_sales.csv", index=False)

print("Dataset saved successfully!")

## Conclusion

The dataset was cleaned by handling missing values, removing duplicates, standardizing values, correcting data types, and detecting outliers. The cleaned dataset was saved as `cleaned_cafe_sales.csv`.